In [2]:
import torch

In [9]:
t1 = torch.tensor([[1,0,1,1],[1,1,0,1]])

In [12]:
t2 = torch.cat((t1,t1,t1), dim = 0)

In [13]:
t2

tensor([[1, 0, 1, 1],
        [1, 1, 0, 1],
        [1, 0, 1, 1],
        [1, 1, 0, 1],
        [1, 0, 1, 1],
        [1, 1, 0, 1]])

In [14]:
t3 = t2 @ t2.T

In [15]:
t3

tensor([[3, 2, 3, 2, 3, 2],
        [2, 3, 2, 3, 2, 3],
        [3, 2, 3, 2, 3, 2],
        [2, 3, 2, 3, 2, 3],
        [3, 2, 3, 2, 3, 2],
        [2, 3, 2, 3, 2, 3]])

In [16]:
t3.shape

torch.Size([6, 6])

In [17]:
t3.dtype

torch.int64

In [18]:
t3.device

device(type='cpu')

In [26]:
t4 = torch.rand(t3.shape)

In [27]:
t4

tensor([[0.1014, 0.1091, 0.1312, 0.2654, 0.1120, 0.7273],
        [0.8005, 0.0989, 0.7841, 0.2379, 0.0432, 0.3968],
        [0.4955, 0.3804, 0.2894, 0.7260, 0.3813, 0.5080],
        [0.3577, 0.1254, 0.8559, 0.5339, 0.8871, 0.9105],
        [0.7700, 0.6151, 0.7919, 0.8205, 0.6306, 0.6662],
        [0.0923, 0.3169, 0.6851, 0.4578, 0.9645, 0.9081]])

In [34]:
total = t4.sum()

In [35]:
total.item()

53.977935791015625

In [33]:
t4.add_(1)

tensor([[1.1014, 1.1091, 1.1312, 1.2654, 1.1120, 1.7273],
        [1.8005, 1.0989, 1.7841, 1.2379, 1.0432, 1.3968],
        [1.4955, 1.3804, 1.2894, 1.7260, 1.3813, 1.5080],
        [1.3577, 1.1254, 1.8559, 1.5339, 1.8871, 1.9105],
        [1.7700, 1.6151, 1.7919, 1.8205, 1.6306, 1.6662],
        [1.0923, 1.3169, 1.6851, 1.4578, 1.9645, 1.9081]])

Article - https://huggingface.co/blog/dvgodoy/beginner-pytorch-tutorial


In [47]:
a = torch.tensor(1.0,requires_grad=True)

In [48]:
b = torch.tensor(2.0, requires_grad =True)

In [49]:
c = a*b

In [50]:
type(c)

torch.Tensor

In [53]:
c.backward()

In [54]:
a.grad

tensor(2.)

In [55]:
b.grad

tensor(1.)

#### lets make a simple training loop

In [66]:
w = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(2.0)
b = torch.tensor(3.0, requires_grad=True)
y = w*x + b
y.retain_grad() 
activation = torch.tanh(y)
activation.backward()

In [67]:
print(w.grad)
print(b.grad)
print(y.grad)



tensor(0.0004)
tensor(0.0002)
tensor(0.0002)


#### now the MLP in pytorch

In [46]:
import torch
import torch.nn as nn

In [47]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [48]:
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [49]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

In [54]:
test_loader = DataLoader(train_data, batch_size=64, shuffle=True)

In [51]:
class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 128)
        self.layer2 = nn.Linear(128,64)
        self.layer3 = nn.Linear(64,32)
        self.layer4 = nn.Linear(32,10)
        pass
    def forward(self, inputs):
        layer1_output = torch.relu(self.layer1(inputs))
        layer2_output = torch.relu(self.layer2(layer1_output))
        layer3_output = torch.relu(self.layer3(layer2_output))
        layer4_output = self.layer4(layer3_output)
        return layer4_output
        

In [52]:
model = Model()
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01) 

In [55]:
epochs = 10
for epoch in range(epochs):   
    for batch_idx, (images, labels) in enumerate(train_loader):
        output = model.forward(images.reshape(-1,784))
        loss_value = loss_function(output, labels)
        optimizer.zero_grad()
        loss_value.backward()
        optimizer.step()
    # find accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            total+=images.shape[0]
            output = model.forward(images.reshape(-1,784))
            y_pred = torch.argmax(output, dim=1)
            equal_mask = torch.eq(y_pred, labels)
            correct += torch.sum(equal_mask).item()
    print(f"After epoch {epoch}, accuracy is {correct/total}")
    model.train()
    

After epoch 0, accuracy is 0.9130166666666667
After epoch 1, accuracy is 0.92785
After epoch 2, accuracy is 0.9373
After epoch 3, accuracy is 0.9480833333333333
After epoch 4, accuracy is 0.9590333333333333
After epoch 5, accuracy is 0.96275
After epoch 6, accuracy is 0.9675666666666667
After epoch 7, accuracy is 0.9698666666666667
After epoch 8, accuracy is 0.9739166666666667
After epoch 9, accuracy is 0.9747666666666667
